In [1]:
from datasets import load_dataset, concatenate_datasets
dataset_names = ["gsm8k", "math_dataset", "dapo_math_17k"]
model = "Qwen3-4B"
user = "divelab"
dataset_paths = [f"{user}/{dataset_name}_{model}_sft" for dataset_name in dataset_names]
ds_all = [load_dataset(dataset, split="train") for dataset in dataset_paths]
ds = concatenate_datasets(ds_all)
df = ds.to_pandas()
df.columns
# ['source_index', 'chunk_id', 'source_parquet_chunk', 'response_idx',
#        'prompt', 'response', 'response_num_tokens', 'response_hit_max_tokens',
#        'messages', 'data_source', 'ground_truth', 'gold_answer', 'is_correct',
#        'extracted_answer']
def analyze_dataset(_dataset_df):
    is_correct = _dataset_df['is_correct'] == 1
    response_hit_max_tokens = _dataset_df['response_hit_max_tokens']
    num_tokens = _dataset_df['response_num_tokens']
    num_correct_tokens = num_tokens[is_correct]
    accuracy = is_correct.mean()
    hit_max_tokens = response_hit_max_tokens.mean()
    mean_correct_tokens_per_response = num_correct_tokens.mean()
    total_tokens = num_tokens.sum()
    total_correct_tokens = num_correct_tokens.sum()
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Hit Max Tokens (2048): {hit_max_tokens:.4f}")
    print(f"Mean Tokens per Response: {num_tokens.mean():.2f}")
    print(f"Mean Tokens per Correct Response: {mean_correct_tokens_per_response:.2f}")
    print(f"Total Tokens: {total_tokens:,}")
    print(f"Total Correct Tokens: {total_correct_tokens:,}")    
print("---" * 20)
print(f"Overall:")
analyze_dataset(df)
for dataset_name in dataset_names:
    print("---" * 20)
    print(f"{dataset_name}")
    dataset_df = df[df['data_source'] == dataset_name]
    analyze_dataset(dataset_df)


/scratch/user/atharvchagi_tamu.edu/.conda/envs/dllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 141160/141160 [00:03<00:00, 46252.67 examples/s]


------------------------------------------------------------
Overall:
Accuracy: 0.5990
Hit Max Tokens (2048): 0.2136
Mean Tokens per Response: 1020.25
Mean Tokens per Correct Response: 630.25
Total Tokens: 296,780,423
Total Correct Tokens: 109,815,218
------------------------------------------------------------
gsm8k
Accuracy: 0.9380
Hit Max Tokens (2048): 0.0007
Mean Tokens per Response: 296.71
Mean Tokens per Correct Response: 286.80
Total Tokens: 22,173,065
Total Correct Tokens: 20,103,521
------------------------------------------------------------
math_dataset
Accuracy: 0.6376
Hit Max Tokens (2048): 0.1057
Mean Tokens per Response: 820.24
Mean Tokens per Correct Response: 601.19
Total Tokens: 61,517,822
Total Correct Tokens: 28,749,673
------------------------------------------------------------
dapo_math_17k
Accuracy: 0.3990
Hit Max Tokens (2048): 0.3835
Mean Tokens per Response: 1509.56
Mean Tokens per Correct Response: 1082.33
Total Tokens: 213,089,536
Total Correct Tokens: 60,

In [2]:
from datasets import load_dataset, load_from_disk
from pathlib import Path

hf_paths = [
    "divelab/gsm8k_Qwen3-4B_sft",
    "divelab/math_dataset_Qwen3-4B_sft",
    "divelab/dapo_math_17k_Qwen3-4B_sft",
]
local_preprocessed_path = Path(
    "/scratch/user/atharvchagi_tamu.edu/dllm_fork/data/sft/a2d/qwen3-0.6b/sft-full-mix-correct-only"
)

required_columns = {"input_ids", "labels"}

print("=" * 90)
print("Step 1: Count correct rows from raw HF datasets")
expected_total_correct = 0

for p in hf_paths:
    raw_ds = load_dataset(p, split="train")
    correct_count = sum(1 for row in raw_ds if row.get("is_correct") == 1)
    expected_total_correct += correct_count
    print(f"{p}: correct_rows={correct_count:,} / total_rows={len(raw_ds):,}")

print(f"Expected total correct rows (HF): {expected_total_correct:,}")

print("\n" + "=" * 90)
print("Step 2: Load local preprocessed dataset")
print(f"Local path: {local_preprocessed_path}")
if not local_preprocessed_path.exists():
    raise FileNotFoundError(f"Dataset path not found: {local_preprocessed_path}")

local_ds = load_from_disk(str(local_preprocessed_path))
print(f"Loaded local object type: {type(local_ds).__name__}")

if hasattr(local_ds, "keys"):
    split_names = list(local_ds.keys())
    local_total_rows = sum(len(local_ds[s]) for s in split_names)
else:
    split_names = ["train"]
    local_total_rows = len(local_ds)

print(f"Local splits: {split_names}")
print(f"Local total rows: {local_total_rows:,}")

print("\n" + "=" * 90)
print("Step 3: Validate local tokenized schema/integrity")

schema_ok = True
seq_ok = True
for split_name in split_names:
    split_ds = local_ds[split_name] if hasattr(local_ds, "keys") else local_ds
    cols = set(split_ds.column_names)
    missing = sorted(required_columns - cols)

    print(f"[{split_name}] rows={len(split_ds):,}, columns={split_ds.column_names}")
    if missing:
        schema_ok = False
        print(f"  MISSING required columns: {missing}")
        continue

    bad_len_pairs = 0
    empty_sequences = 0

    for row in split_ds:
        input_ids = row["input_ids"]
        labels = row["labels"]
        if len(input_ids) != len(labels):
            bad_len_pairs += 1
        if len(input_ids) == 0:
            empty_sequences += 1

    print(f"  length_mismatches={bad_len_pairs}, empty_sequences={empty_sequences}")
    if bad_len_pairs > 0 or empty_sequences > 0:
        seq_ok = False

print("\n" + "=" * 90)
print("Step 4: Verify correct-only guarantee by row-count equivalence")
count_match = local_total_rows == expected_total_correct

if count_match:
    print(
        f"PASS: local_total_rows ({local_total_rows:,}) == expected_total_correct ({expected_total_correct:,})"
    )
else:
    delta = local_total_rows - expected_total_correct
    print(
        f"FAIL: local_total_rows ({local_total_rows:,}) != expected_total_correct ({expected_total_correct:,}); delta={delta:,}"
    )

overall_ok = count_match and schema_ok and seq_ok
print("\n" + "#" * 90)
print(f"OVERALL CHECK: {'PASS' if overall_ok else 'FAIL'}")
print(f"- Count match: {'PASS' if count_match else 'FAIL'}")
print(f"- Schema check: {'PASS' if schema_ok else 'FAIL'}")
print(f"- Sequence check: {'PASS' if seq_ok else 'FAIL'}")

Step 1: Count correct rows from raw HF datasets
divelab/gsm8k_Qwen3-4B_sft: correct_rows=70,096 / total_rows=74,730
divelab/math_dataset_Qwen3-4B_sft: correct_rows=47,821 / total_rows=75,000
divelab/dapo_math_17k_Qwen3-4B_sft: correct_rows=56,325 / total_rows=141,160
Expected total correct rows (HF): 174,242

Step 2: Load local preprocessed dataset
Local path: /scratch/user/atharvchagi_tamu.edu/dllm_fork/data/sft/a2d/qwen3-0.6b/sft-full-mix-correct-only
Loaded local object type: DatasetDict
Local splits: ['train']
Local total rows: 174,242

Step 3: Validate local tokenized schema/integrity
[train] rows=174,242, columns=['input_ids', 'labels', 'prompt_len']
  length_mismatches=0, empty_sequences=0

Step 4: Verify correct-only guarantee by row-count equivalence
PASS: local_total_rows (174,242) == expected_total_correct (174,242)

##########################################################################################
OVERALL CHECK: PASS
- Count match: PASS
- Schema check: PASS
- Sequen